In [2]:
import sqlite3#para la base de datos
import requests#para realizar las peticiones
from bs4 import BeautifulSoup#para parsear el html
import time#para evitar mandar muchas peticiones 
from concurrent.futures import ThreadPoolExecutor, as_completed#para relizar un scraping veloz
import pandas as pd#para visualizar las consultas

In [3]:
BASE_URL = "http://books.toscrape.com/"#url base donde se realiza el scraping
API_SEARCH_URL = "https://openlibrary.org/search.json"#api donde consultamos los datos necesario para el enriqucimiento
API_AUTHOR_URL = "https://openlibrary.org/authors/{}.json"#hace un marcador de posicion que utilizaremos mas adelante para el id
author_cache = {}#aca se guarda la el cache para evitar llamadas innecesarias a la api
# Mapeo de ratings de texto a números
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}#en el html de la pagina la calififcacion esta en str aca le damos un valor int

In [4]:

def crear_base_datos():
    conn = sqlite3.connect('books_detective.db')
    cursor = conn.cursor()
    
    # DDL: Creación de tablas
    cursor.executescript('''
        CREATE TABLE IF NOT EXISTS categories (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE
        );

        CREATE TABLE IF NOT EXISTS authors (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            birth_year TEXT,
            country TEXT,
            external_api_id TEXT,
            total_known_works INTEGER,
            api_source TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT UNIQUE,
            price REAL,
            rating INTEGER,
            category_id INTEGER,
            FOREIGN KEY (category_id) REFERENCES categories (id)
        );

        CREATE TABLE IF NOT EXISTS book_author (
            book_id INTEGER,
            author_id INTEGER,
            PRIMARY KEY (book_id, author_id),
            FOREIGN KEY (book_id) REFERENCES books (id),
            FOREIGN KEY (author_id) REFERENCES authors (id)
        );
    ''')
    conn.commit()
    conn.close()

crear_base_datos()

In [ ]:
def obtener_datos_autor_api(book_title, max_reintentos=3):#aca realizamos el enriquecimiento
    
    for intento in range(max_reintentos):#limitamos los reintentos a 3 para no sobrecargar
        try:
            # Paso 1: Buscar el libro (Timeout aumentado a 20)
            res_search = requests.get(API_SEARCH_URL, params={'title': book_title, 'limit': 1}, timeout=20)#hacemos una petcion a la api con limite de 1 y 20 segundos de espera
            
            # Si el servidor nos bloquea por muchas peticiones (Error 429), esperamos más.
            if res_search.status_code == 429:
                print("⚠️ API saturada (Error 429). Esperando 5 segundos...")
                time.sleep(5)
                continue # Pasa al siguiente intento del bucle
                
            if res_search.status_code != 200: #dudas
                return None
            
            data = res_search.json()#transformamos en un objeto nativo en python
            if not data.get('docs') or not data['docs'][0].get('author_key'):#retornamos none en caso de que este vacio o que no encuentre el id
                return None
                
            author_key = data['docs'][0]['author_key'][0]#obtenemos el id del autor
            author_name = data['docs'][0].get('author_name', ['Unknown'])[0]#en base al libro buscamos la etiqueta de autor name
            
            # Revisar Caché
            if author_key in author_cache:#evitamos llamadas innecesarias a la api
                return author_cache[author_key]

            # Paso 2: Obtener detalles del autor
            time.sleep(1) # Respetar rate limits
            res_author = requests.get(API_AUTHOR_URL.format(author_key), timeout=20)#realizamos una busqueda en base a el id del autor 
            if res_author.status_code != 200: return None#retornamos none en caso de no tener exito con la peticion
            
            a_data = res_author.json()#transformamos en un objeto nativo de python el html
            
            birth_date = a_data.get('birth_date', None)#buscamos el anho de nacimiento
            country = a_data.get('location', None) #buscamos la nacionalidad
            
            author_info = {#retornamos los datos que ya encontramos 
                "name": author_name,
                "birth_year": str(birth_date)[-4:] if birth_date else None,#dudas
                "country": str(country) if country else "Unknown",#dudas
                "external_api_id": author_key,
                "total_known_works": None,
                "api_source": "OpenLibrary"
            }
            
            author_cache[author_key] = author_info#hacemos un expediente de la informacion
            return author_info#retornamos los valores

        except requests.exceptions.Timeout:#si se supero el timeout de 20 indicamos que vamos a reintentar con un descanso de 2 segundos
            print(f"🕒 Timeout buscando '{book_title}' (Intento {intento + 1} de {max_reintentos}). Reintentando...")#indicamos los reintentos realizados
            time.sleep(2) # Espera 2 segundos antes de volver hacer una peticion
            
        except requests.exceptions.RequestException as e:#visualizamos el error de nuestra peticion
            print(f"❌ Error crítico de red con API: {e}")
            return None 
            
    # Si termina el bucle for y no logró conectarse:
    print(f"🛑 Se agotaron los reintentos para el libro '{book_title}'.")#dudas
    return None

In [ ]:
def procesar_libro(book_url):#scrapin principal a la pagina de bookstoscrap
    try:
        res = requests.get(book_url)#hacemos una peticion a esta url
        soup = BeautifulSoup(res.content, 'html.parser') #transformamos en binario y parseamos
        
        # Scrapeo básico del titulo,precio,rating,categoria
        title = soup.find('h1').text
        price_str = soup.find('p', class_='price_color').text
        price = float(price_str.replace('£', '').replace('Â', '').strip())
        rating_class = soup.find('p', class_='star-rating')['class'][1]
        rating = RATING_MAP.get(rating_class, 0)
        
        breadcrumb = soup.find('ul', class_='breadcrumb')
        category_name = breadcrumb.find_all('li')[2].text.strip()
        
        # Llamar a API
        author_data = obtener_datos_autor_api(title)#pasamos el titulo del libro por la funcion de la api para enriquecer
        
        return {
            "title": title, "price": price, "rating": rating, 
            "category_name": category_name, "author": author_data
        }
    except Exception as e:#indicamos los errores
        print(f"Error procesando {book_url}: {e}")
        return None

In [ ]:
def recopilar_todas_las_urls():#guaramos los links de la pagina
    urls = []
    print("Buscando las URLs de los 1000 libros... (esto toma unos segundos)")
    for i in range(1, 51): # Sabemos que el sitio tiene 50 páginas
        url_pagina = f"http://books.toscrape.com/catalogue/page-{i}.html"
        res = requests.get(url_pagina)
        soup = BeautifulSoup(res.content, 'html.parser')
        
        for articulo in soup.find_all('article', class_='product_pod'):#iteramos en todos las etiquetas con ese contenido
            href = articulo.find('h3').find('a')['href']#guaramos el link de referencia
            # Normalizamos el link relativo
            href_limpio = href if not href.startswith('../') else href.replace('../', '')#hacemos una limpieza  del link para evitar errores
            url_completa = BASE_URL + "catalogue/" + href_limpio#juntamos todo en un solo link
            urls.append(url_completa)#agregamos a la lista de urls
            
    return urls

def orquestador_scraping(limite_libros=None):    
    # 1. Obtener todas las URLs reales
    todas_las_urls = recopilar_todas_las_urls()    
    print(f"Comenzando el scraping de {len(todas_las_urls)} libros con 5 hilos...")

    resultados = []
    # 2. IMPLEMENTACIÓN DE HILOS
    with ThreadPoolExecutor(max_workers=5) as executor:#ejecutamos el scraping multihilos
        # Ahora pasamos solo la URL, ya que la función saca la categoría por sí misma
        futuros = {executor.submit(procesar_libro, url): url for url in todas_las_urls}#guardamos el proceso como promesa de resultado
        
        for futuro in as_completed(futuros):#iteramos entre todas las promesas
            data = futuro.result()#extraemos los resultados de las promesas
            if data: resultados.append(data)#si no esta vacio agregamos a la lista resultados
            
    return resultados

In [ ]:
def guardar_resultados_en_db(lista_resultados, db_name='books_detective.db'):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    for item in lista_resultados:
        try:
            # 1. Manejo de la Categoría
            cursor.execute("INSERT OR IGNORE INTO categories (name) VALUES (?)", (item['category_name'],))#guaradmos la categoria
            cursor.execute("SELECT id FROM categories WHERE name = ?", (item['category_name'],))#buscamos la categoria que coincida con el id
            category_id = cursor.fetchone()[0]#vinculamos la categoria con el id

            # 2. Manejo del Autor (Enriquecido por API)
            author_id = None
            if item['author']:
                auth = item['author']
                cursor.execute("""
                    INSERT OR IGNORE INTO authors 
                    (name, birth_year, country, external_api_id, total_known_works, api_source)
                    VALUES (?, ?, ?, ?, ?, ?)
                """, (auth['name'], auth['birth_year'], auth['country'], 
                      auth['external_api_id'], auth['total_known_works'], auth['api_source']))
                
                cursor.execute("SELECT id FROM authors WHERE name = ?", (auth['name'],))
                author_id = cursor.fetchone()[0]#vinculamos el autor con su id

            # 3. Manejo del Libro
            cursor.execute("""
                INSERT OR IGNORE INTO books (title, price, rating, category_id)
                VALUES (?, ?, ?, ?)
            """, (item['title'], item['price'], item['rating'], category_id))#duda
            
            cursor.execute("SELECT id FROM books WHERE title = ?", (item['title'],))
            book_id = cursor.fetchone()[0]#vinculamos el id del libro con el libro

            # 4. Vincular Libro y Autor (Tabla muchos a muchos)
            if author_id:
                cursor.execute("INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)", 
                               (book_id, author_id))

        except Exception as e:
            print(f"Error insertando el libro '{item['title']}': {e}")
            conn.rollback()#ligada a la atomicidad si algo esta a medias no entra por cualquier motivo
            continue

    conn.commit()
    conn.close()
    print(f"✅ Proceso terminado. Se intentaron procesar {len(lista_resultados)} registros.")

In [ ]:
def main():#acasucede la union de todo ejecutamos la base de datos, realizamos el scraping e insertamos los datos del scraping y el enriquecimiento
    print("🕵️‍♂️ Iniciando investigación masiva en Books To Scrape...")
    start_time = time.time()

    # Paso 1: Preparar la Base de Datos
    print("[1/3] Verificando esquemas de la base de datos SQLite...")
    crear_base_datos()

   
    print("[2/3] Mapeando sitio, lanzando hilos y consultando API...")
    resultados_crudos = orquestador_scraping() 

    # Paso 3: Persistencia en Disco
    if resultados_crudos:
        print(f"[3/3] Archivando {len(resultados_crudos)} expedientes en la base de datos...")
        guardar_resultados_en_db(resultados_crudos)
    else:
        print("❌ Misión fallida: No se extrajeron datos. Revisa la conexión o los selectores HTML.")

    end_time = time.time()
    print(f"🏁 Ejecución total finalizada en {round(end_time - start_time, 2)} segundos.")

# Punto de entrada estándar en Python
if __name__ == '__main__':
    main()

🕵️‍♂️ Iniciando investigación masiva en Books To Scrape...
[1/3] Verificando esquemas de la base de datos SQLite...
[2/3] Mapeando sitio, lanzando hilos y consultando API...
Buscando las URLs de los 1000 libros... (esto toma unos segundos)
Comenzando el scraping de 1000 libros con 5 hilos...
🕒 Timeout buscando 'A Light in the Attic' (Intento 1 de 3). Reintentando...
🕒 Timeout buscando 'The Dirty Little Secrets of Getting Your Dream Job' (Intento 1 de 3). Reintentando...
🕒 Timeout buscando 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull' (Intento 1 de 3). Reintentando...
🕒 Timeout buscando 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics' (Intento 1 de 3). Reintentando...
🕒 Timeout buscando 'A Light in the Attic' (Intento 2 de 3). Reintentando...
🕒 Timeout buscando 'Aladdin and His Wonderful Lamp' (Intento 1 de 3). Reintentando...
🕒 Timeout buscando 'The Elephant Tree' (Intento 1 de 3). Reintentando.

In [5]:
conn = sqlite3.connect('books_detective.db')

print("🔍 1. Libros con más de 3 estrellas por menos de £10:")
query_1 = """
SELECT title, price, rating 
FROM books 
WHERE rating > 3 AND price < 10.0;
"""
df_1 = pd.read_sql_query(query_1, conn)
display(df_1.head()) # Muestra los primeros 5 resultados

print("\n📉 2. Autor con peor promedio de rating (mínimo 5 libros):")
query_2 = """
SELECT a.name, AVG(b.rating) as avg_rating, COUNT(b.id) as total_books
FROM authors a
JOIN book_author ba ON a.id = ba.author_id
JOIN books b ON ba.book_id = b.id
GROUP BY a.id
HAVING total_books >= 5
ORDER BY avg_rating ASC
LIMIT 1;
"""
df_2 = pd.read_sql_query(query_2, conn)
display(df_2)

print("\n💰 3. Categoría con mayor precio promedio:")
query_3 = """
SELECT c.name, AVG(b.price) as avg_price
FROM categories c
JOIN books b ON c.id = b.category_id
GROUP BY c.id
ORDER BY avg_price DESC
LIMIT 1;
"""
df_3 = pd.read_sql_query(query_3, conn)
display(df_3)

print("\n🏆 4. Top 5 autores con más libros:")
query_4 = """
SELECT a.name, COUNT(ba.book_id) as book_count
FROM authors a
JOIN book_author ba ON a.id = ba.author_id
GROUP BY a.id
ORDER BY book_count DESC
LIMIT 5;
"""
df_4 = pd.read_sql_query(query_4, conn)
display(df_4)

print("\n🌍 5. OBLIGATORIA: País que produce más libros con rating > 3:")
query_5 = """
SELECT a.country, COUNT(b.id) as good_books_count
FROM books b
JOIN book_author ba ON b.id = ba.book_id
JOIN authors a ON ba.author_id = a.id
WHERE b.rating > 3 AND a.country IS NOT NULL AND a.country != 'Unknown'
GROUP BY a.country
ORDER BY good_books_count DESC
LIMIT 1;
"""
df_5 = pd.read_sql_query(query_5, conn)
display(df_5)

🔍 1. Libros con más de 3 estrellas por menos de £10:


,title,price,rating



📉 2. Autor con peor promedio de rating (mínimo 5 libros):


,name,avg_rating,total_books
0,Worth Books,2.3,10



💰 3. Categoría con mayor precio promedio:


,name,avg_price
0,Suspense,58.33



🏆 4. Top 5 autores con más libros:


,name,book_count
0,Worth Books,10
1,Stephen King,9
2,David Levithan,4
3,Sophie Kinsella,4
4,David Sedaris,4



🌍 5. OBLIGATORIA: País que produce más libros con rating > 3:


,country,good_books_count
0,/authors/OL39872A,1


In [6]:
import time

cursor = conn.cursor()

# 1. Asegurarnos de limpiar índices previos si corres esta celda varias veces
cursor.execute("DROP INDEX IF EXISTS idx_books_rating;")
cursor.execute("DROP INDEX IF EXISTS idx_authors_country;")
conn.commit()

# Consulta deliberadamente pesada
query_rendimiento = """
SELECT a.country, COUNT(b.id) as good_books_count
FROM books b
JOIN book_author ba ON b.id = ba.book_id
JOIN authors a ON ba.author_id = a.id
WHERE b.rating > 3 AND a.country IS NOT NULL AND a.country != 'Unknown'
GROUP BY a.country
ORDER BY good_books_count DESC;
"""

# 2. Medición SIN índices (Full Table Scan)
start_time = time.perf_counter()#capturamos el tiempo exacto antes de ejecutar
pd.read_sql_query(query_rendimiento, conn)#ejecutamos
tiempo_sin_indice = time.perf_counter() - start_time#calculamos el tiempo transcurrido
print(f"🐌 Tiempo de ejecución SIN índices: {tiempo_sin_indice:.6f} segundos")#vemos el resultado

# 3. Creación de Índices
print("🧱 Creando índices B-Tree en rating y country...")
cursor.execute("CREATE INDEX idx_books_rating ON books(rating);")
cursor.execute("CREATE INDEX idx_authors_country ON authors(country);")
conn.commit()

# 4. Medición CON índices (Index Scan)
start_time = time.perf_counter()
pd.read_sql_query(query_rendimiento, conn)
tiempo_con_indice = time.perf_counter() - start_time
print(f"🚀 Tiempo de ejecución CON índices: {tiempo_con_indice:.6f} segundos")

# Explicación de los resultados
mejora = (tiempo_sin_indice / tiempo_con_indice) if tiempo_con_indice > 0 else 0
print(f"📈 La consulta fue aproximadamente {mejora:.2f} veces más rápida.")
conn.close()

🐌 Tiempo de ejecución SIN índices: 0.001442 segundos
🧱 Creando índices B-Tree en rating y country...
🚀 Tiempo de ejecución CON índices: 0.001605 segundos
📈 La consulta fue aproximadamente 0.90 veces más rápida.
